# Silver Layer — Suppliers
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.suppliers`, applies cleaning and null filling,
and writes the curated result to `salesflow_dev.silver.suppliers`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Remove duplicates by `SupplierID` |
| 2 | Clean `CompanyName`, `City`, `Country`, `ContactName` |
| 3 | Standardize `Phone` |
| 4 | Fill nulls in optional fields |
| 5 | Add `data_quality_status` flag |
| 6 | Add `processing_timestamp` |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.table("salesflow_dev.bronze.suppliers")
print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Remove Duplicates

In [0]:
df = df.dropDuplicates(["SupplierID"])
print(f"Records after deduplication: {df.count()}")

## 3. Clean and Standardize Columns

In [0]:
# Clean text columns
df = clean_string_column(df, "Name")
df = clean_string_column(df, "City")
df = clean_string_column(df, "source_system")



## 4. Add Quality Flag
`INVALID` if `SupplierID` or `CompanyName` is null.

In [0]:
df = add_quality_flag(df, ["SupplierID"])

## 5. Add Processing Timestamp

In [0]:
df = df.withColumn("processing_timestamp", current_timestamp())

## 6. Save as Delta Table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("salesflow_dev.silver.suppliers")
print("Table saved: salesflow_dev.silver.suppliers")

## 7. Validation

In [0]:
silver_suppliers = spark.table("salesflow_dev.silver.suppliers")
print(f"Total records: {silver_suppliers.count()}")
print("\nQuality flag distribution:")
display(silver_suppliers.groupBy("data_quality_status").count())
print("\nSchema:")
silver_suppliers.printSchema()
print("\nFirst 5 rows:")
display(silver_suppliers.limit(5))